In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
import time

sp500_df = pd.read_csv("../data/raw/sp500_tickers.csv")
tickers = sp500_df['Symbol'].tolist()

print(f"Loaded {len(tickers)} tickers")

Loaded 503 tickers


In [2]:
def extract_company_data(ticker_symbol):
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info

        # --- directly from yfinance ---
        data = {
            'symbol': ticker_symbol,
            'name': info.get('longName'),
            'sector': info.get('sector'),
            'industry': info.get('industry'),
            'current_price': info.get('currentPrice'),
            'market_cap': info.get('marketCap'),
            'enterprise_value': info.get('enterpriseValue'),
            'pe_ratio': info.get('trailingPE'),
            'forward_pe': info.get('forwardPE'),
            'ps_ratio': info.get('priceToSalesTrailing12Months'),
            'pb_ratio': info.get('priceToBook'),
            'ev_ebitda': info.get('enterpriseToEbitda'),
            'ev_revenue': info.get('enterpriseToRevenue'),
            'peg_ratio': info.get('pegRatio'),
            'beta': info.get('beta'),
            'revenue': info.get('totalRevenue'),
            'gross_profit': info.get('grossProfits'),
            'ebitda': info.get('ebitda'),
            'net_income': info.get('netIncomeToCommon'),
            'earnings_per_share': info.get('trailingEps'),
            'free_cash_flow': info.get('freeCashflow'),
            'operating_cash_flow': info.get('operatingCashflow'),
            'total_debt': info.get('totalDebt'),
            'total_cash': info.get('totalCash'),
            'shares_outstanding': info.get('sharesOutstanding'),
            'roe': info.get('returnOnEquity'),
            'roa': info.get('returnOnAssets'),
            'debt_to_equity': info.get('debtToEquity'),
            'current_ratio': info.get('currentRatio'),
            'quick_ratio': info.get('quickRatio'),
            'fifty_two_week_high': info.get('fiftyTwoWeekHigh'),
            'fifty_two_week_low': info.get('fiftyTwoWeekLow'),
            'dividend_yield': info.get('dividendYield'),
            'payout_ratio': info.get('payoutRatio'),
            'analyst_target_price': info.get('targetMeanPrice'),
            'recommendation': info.get('recommendationKey'),
        }

        # --- calculated metrics ---

        # margins
        if data['revenue'] and data['revenue'] != 0:
            data['gross_margin'] = data['gross_profit'] / data['revenue'] if data['gross_profit'] else None
            data['ebitda_margin'] = data['ebitda'] / data['revenue'] if data['ebitda'] else None
            data['net_margin'] = data['net_income'] / data['revenue'] if data['net_income'] else None
            data['fcf_margin'] = data['free_cash_flow'] / data['revenue'] if data['free_cash_flow'] else None
        else:
            data['gross_margin'] = None
            data['ebitda_margin'] = None
            data['net_margin'] = None
            data['fcf_margin'] = None

        # 52 week position (where is current price between 52w low and high, 0-100%)
        if data['fifty_two_week_high'] and data['fifty_two_week_low'] and data['current_price']:
            range_size = data['fifty_two_week_high'] - data['fifty_two_week_low']
            if range_size != 0:
                data['fifty_two_week_position'] = (data['current_price'] - data['fifty_two_week_low']) / range_size * 100
            else:
                data['fifty_two_week_position'] = None
        else:
            data['fifty_two_week_position'] = None

        # net debt
        if data['total_debt'] is not None and data['total_cash'] is not None:
            data['net_debt'] = data['total_debt'] - data['total_cash']
        else:
            data['net_debt'] = None

        # price to free cash flow
        if data['free_cash_flow'] and data['shares_outstanding'] and data['current_price']:
            fcf_per_share = data['free_cash_flow'] / data['shares_outstanding']
            if fcf_per_share != 0:
                data['p_fcf'] = data['current_price'] / fcf_per_share
            else:
                data['p_fcf'] = None
        else:
            data['p_fcf'] = None

        # altman z-score (requires balance sheet data)
        try:
            bs = pd.read_csv(f"../data/raw/{ticker_symbol}/balance_sheet.csv", index_col=0)
            is_ = pd.read_csv(f"../data/raw/{ticker_symbol}/income_statement.csv", index_col=0)

            def get_val(df, possible_names):
                for name in possible_names:
                    if name in df.index:
                        val = df.loc[name].iloc[0]
                        if pd.notna(val):
                            return float(val)
                return None

            total_assets = get_val(bs, ['Total Assets'])
            total_liabilities = get_val(bs, ['Total Liabilities Net Minority Interest'])
            current_assets = get_val(bs, ['Current Assets'])
            current_liabilities = get_val(bs, ['Current Liabilities'])
            retained_earnings = get_val(bs, ['Retained Earnings'])
            ebit = get_val(is_, ['EBIT'])

            if all(v is not None for v in [total_assets, total_liabilities, current_assets, current_liabilities, retained_earnings, ebit]):
                working_capital = current_assets - current_liabilities
                book_equity = total_assets - total_liabilities
                market_cap_val = data['market_cap'] if data['market_cap'] else 0
                revenue_val = data['revenue'] if data['revenue'] else 0

                if total_assets != 0:
                    x1 = working_capital / total_assets
                    x2 = retained_earnings / total_assets
                    x3 = ebit / total_assets
                    x4 = market_cap_val / total_liabilities if total_liabilities != 0 else 0
                    x5 = revenue_val / total_assets

                    z_score = 1.2*x1 + 1.4*x2 + 3.3*x3 + 0.6*x4 + 1.0*x5
                    data['altman_z_score'] = round(z_score, 3)

                    if z_score > 3.0:
                        data['z_score_zone'] = 'safe'
                    elif z_score > 1.8:
                        data['z_score_zone'] = 'grey'
                    else:
                        data['z_score_zone'] = 'distress'
                else:
                    data['altman_z_score'] = None
                    data['z_score_zone'] = None
            else:
                data['altman_z_score'] = None
                data['z_score_zone'] = None

        except Exception:
            data['altman_z_score'] = None
            data['z_score_zone'] = None

        return data

    except Exception as e:
        print(f"  {ticker_symbol} failed: {e}")
        return None

In [3]:
apple_data = extract_company_data("AAPL")

for key, value in apple_data.items():
    print(f"{key}: {value}")

symbol: AAPL
name: Apple Inc.
sector: Technology
industry: Consumer Electronics
current_price: 311.95
market_cap: 4581720850432
enterprise_value: 4587349606400
pe_ratio: 37.812122
forward_pe: 32.468006
ps_ratio: 10.149079
pb_ratio: 42.96832
ev_ebitda: 28.675
ev_revenue: 10.162
peg_ratio: 2.53
beta: 1.086
revenue: 451442016256
gross_profit: 216070995968
ebitda: 159975997440
net_income: 122575003648
earnings_per_share: 8.25
free_cash_flow: 101090746368
operating_cash_flow: 140222005248
total_debt: 84710998016
total_cash: 68507000832
shares_outstanding: 14687356000
roe: 1.4147099
roa: 0.26229
debt_to_equity: 79.548
current_ratio: 1.07
quick_ratio: 0.906
fifty_two_week_high: 316.94
fifty_two_week_low: 195.07
dividend_yield: 0.35
payout_ratio: 0.1259
analyst_target_price: 310.507
recommendation: buy
gross_margin: 0.4786240274221003
ebitda_margin: 0.3543666554716124
net_margin: 0.2715188202120983
fcf_margin: 0.2239285284218523
fifty_two_week_position: 95.905473045048
net_debt: 16203997184
p_

In [4]:
all_companies = []
failed = []

for i, ticker_symbol in enumerate(tickers):
    print(f"[{i+1}/503] {ticker_symbol}...", end=" ")
    result = extract_company_data(ticker_symbol)
    if result:
        all_companies.append(result)
        print("done")
    else:
        failed.append(ticker_symbol)
        print("failed")
    time.sleep(1)

master_df = pd.DataFrame(all_companies)
master_df.to_csv("../data/processed/master_features.csv", index=False)

print(f"\nDone. {len(all_companies)} companies saved, {len(failed)} failed.")
print(f"Failed: {failed}")

[1/503] MMM... done
[2/503] AOS... done
[3/503] ABT... done
[4/503] ABBV... done
[5/503] ACN... done
[6/503] ADBE... done
[7/503] AMD... done
[8/503] AES... done
[9/503] AFL... done
[10/503] A... done
[11/503] APD... done
[12/503] ABNB... done
[13/503] AKAM... done
[14/503] ALB... done
[15/503] ARE... done
[16/503] ALGN... done
[17/503] ALLE... done
[18/503] LNT... done
[19/503] ALL... done
[20/503] GOOGL... done
[21/503] GOOG... done
[22/503] MO... done
[23/503] AMZN... done
[24/503] AMCR... done
[25/503] AEE... done
[26/503] AEP... done
[27/503] AXP... done
[28/503] AIG... done
[29/503] AMT... done
[30/503] AWK... done
[31/503] AMP... done
[32/503] AME... done
[33/503] AMGN... done
[34/503] APH... done
[35/503] ADI... done
[36/503] AON... done
[37/503] APA... done
[38/503] APO... done
[39/503] AAPL... done
[40/503] AMAT... done
[41/503] APP... done
[42/503] APTV... done
[43/503] ACGL... done
[44/503] ADM... done
[45/503] ARES... done
[46/503] ANET... done
[47/503] AJG... done
[48/503

In [5]:
master_df = pd.read_csv("../data/processed/master_features.csv")

print("Shape:", master_df.shape)
print("\nColumns:", master_df.columns.tolist())
print("\nSample — top 5 by market cap:")
print(master_df.nlargest(5, 'market_cap')[['symbol', 'name', 'market_cap', 'pe_ratio', 'ev_ebitda', 'altman_z_score', 'z_score_zone']])

Shape: (503, 45)

Columns: ['symbol', 'name', 'sector', 'industry', 'current_price', 'market_cap', 'enterprise_value', 'pe_ratio', 'forward_pe', 'ps_ratio', 'pb_ratio', 'ev_ebitda', 'ev_revenue', 'peg_ratio', 'beta', 'revenue', 'gross_profit', 'ebitda', 'net_income', 'earnings_per_share', 'free_cash_flow', 'operating_cash_flow', 'total_debt', 'total_cash', 'shares_outstanding', 'roe', 'roa', 'debt_to_equity', 'current_ratio', 'quick_ratio', 'fifty_two_week_high', 'fifty_two_week_low', 'dividend_yield', 'payout_ratio', 'analyst_target_price', 'recommendation', 'gross_margin', 'ebitda_margin', 'net_margin', 'fcf_margin', 'fifty_two_week_position', 'net_debt', 'p_fcf', 'altman_z_score', 'z_score_zone']

Sample — top 5 by market cap:
    symbol                   name    market_cap   pe_ratio  ev_ebitda  \
343   NVDA     NVIDIA Corporation  4.996429e+12  31.542050     31.727   
38    AAPL             Apple Inc.  4.588330e+12  37.866665     28.675   
19   GOOGL          Alphabet Inc.  4.4634

In [6]:
missing_z = master_df[master_df['altman_z_score'].isna()][['symbol', 'name', 'sector']]
print(f"Companies with missing Z-Score: {len(missing_z)}")
print(missing_z.to_string())

Companies with missing Z-Score: 57
    symbol                                    name                  sector
4      ACN                           Accenture plc              Technology
8      AFL                      Aflac Incorporated      Financial Services
14     ARE   Alexandria Real Estate Equities, Inc.             Real Estate
18     ALL                The Allstate Corporation      Financial Services
19   GOOGL                           Alphabet Inc.  Communication Services
20    GOOG                           Alphabet Inc.  Communication Services
26     AXP                American Express Company      Financial Services
27     AIG      American International Group, Inc.      Financial Services
30     AMP              Ameriprise Financial, Inc.      Financial Services
42    ACGL                 Arch Capital Group Ltd.      Financial Services
47     AIZ                          Assurant, Inc.      Financial Services
58     BAC             Bank of America Corporation      Financial

In [7]:
financial_sectors = ['Financial Services', 'Real Estate']

master_df['z_score_zone'] = master_df.apply(
    lambda row: 'not applicable' if row['sector'] in financial_sectors 
    else row['z_score_zone'], axis=1
)

print("Z-Score zone distribution:")
print(master_df['z_score_zone'].value_counts())

Z-Score zone distribution:
z_score_zone
safe              239
not applicable    100
grey               79
distress           76
Name: count, dtype: int64


In [8]:
master_df.to_csv("../data/processed/master_features.csv", index=False)
print("Saved updated master file.")

print("\nMissing value summary (top 15 columns):")
missing = master_df.isnull().sum()
missing_pct = (missing / len(master_df) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False).head(15))

Saved updated master file.

Missing value summary (top 15 columns):
                missing_count  missing_pct
dividend_yield             97         19.3
altman_z_score             57         11.3
debt_to_equity             53         10.5
free_cash_flow             37          7.4
fcf_margin                 37          7.4
p_fcf                      37          7.4
roe                        35          7.0
ebitda                     33          6.6
ev_ebitda                  33          6.6
ebitda_margin              33          6.6
pe_ratio                   27          5.4
current_ratio              23          4.6
quick_ratio                23          4.6
peg_ratio                   9          1.8
beta                        9          1.8


In [9]:
master_df['dividend_yield'] = master_df['dividend_yield'].fillna(0)
master_df['payout_ratio'] = master_df['payout_ratio'].fillna(0)

sector_medians = master_df.groupby('sector')[['pe_ratio', 'ev_ebitda', 'ps_ratio', 
                                               'pb_ratio', 'debt_to_equity',
                                               'current_ratio', 'quick_ratio',
                                               'roe', 'roa']].transform('median')

fill_cols = ['pe_ratio', 'ev_ebitda', 'ps_ratio', 'pb_ratio', 
             'debt_to_equity', 'current_ratio', 'quick_ratio', 'roe', 'roa']

for col in fill_cols:
    master_df[col] = master_df[col].fillna(sector_medians[col])

print("Missing values after cleaning:")
missing_after = master_df.isnull().sum()
print(missing_after[missing_after > 0])

master_df.to_csv("../data/processed/master_features.csv", index=False)
print("\nSaved cleaned master file.")

Missing values after cleaning:
name                        4
sector                      3
industry                    3
current_price               2
market_cap                  2
enterprise_value            4
pe_ratio                    2
forward_pe                  2
ps_ratio                    3
pb_ratio                    2
ev_ebitda                   3
ev_revenue                  4
peg_ratio                   9
beta                        9
revenue                     3
gross_profit                3
ebitda                     33
net_income                  4
earnings_per_share          3
free_cash_flow             37
operating_cash_flow         5
total_debt                  4
total_cash                  4
shares_outstanding          2
roe                         3
roa                         3
debt_to_equity              3
current_ratio               3
quick_ratio                 3
fifty_two_week_high         2
fifty_two_week_low          2
analyst_target_price        6
recommend

In [10]:
distress = master_df[master_df['z_score_zone'] == 'distress'][['symbol', 'name', 'sector', 'altman_z_score', 'debt_to_equity', 'current_ratio']].sort_values('altman_z_score')

print(f"Distress zone companies: {len(distress)}")
print(distress.to_string())

Distress zone companies: 76
    symbol                                          name                  sector  altman_z_score  debt_to_equity  current_ratio
468   VRSN                                VeriSign, Inc.              Technology          -4.291          55.606          0.462
165   SATS                          EchoStar Corporation  Communication Services          -0.751         515.060          0.302
198    FIS  Fidelity National Information Services, Inc.              Technology           0.087         132.328          0.588
7      AES                           The AES Corporation               Utilities           0.454         259.603          0.730
279    KHC                       The Kraft Heinz Company      Consumer Defensive           0.466          50.258          1.195
358   PSKY                Paramount Skydance Corporation  Communication Services           0.502         130.167          1.100
365    PCG                              PG&E Corporation               Utili